## 1. Importações e Configurações de Parâmetros

`data_referencia_calculo` define o limite superior aceito para `data_pedido`. Se estiver vazio, o notebook usa `current_date()`. Se for preenchido, usa a data informada. Isso permite rodar o pipeline em modo diário ou fixar uma data específica para reprocessamento.

`DATA_MIN_OPERACAO` define o limite inferior de datas aceitas. O valor usado é `2018-01-01`, porque a V-Commerce foi fundada em 2018.

In [0]:
import time
import unicodedata
import re
from datetime import datetime

from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType, TimestampType

dbutils.widgets.text("catalogo", "workspace")
catalogo = dbutils.widgets.get("catalogo")

spark.sql(f"USE CATALOG {catalogo}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalogo}.silver")

execucao_id = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"Catálogo em uso: {catalogo}")
print(f"Execução: {execucao_id}")


dbutils.widgets.text("data_referencia_calculo", "2026-05-22")
data_referencia_param = dbutils.widgets.get("data_referencia_calculo").strip()

DATA_MIN_OPERACAO = "2018-01-01"

if data_referencia_param:
    DATA_REFERENCIA_COL = F.to_date(F.lit(data_referencia_param))
    print(f"Data de referência: {data_referencia_param}")
else:
    DATA_REFERENCIA_COL = F.current_date()
    print("Data de referência: current_date()")


## 2. Funções Utilitárias

Funções auxiliares usadas em diferentes blocos do notebook.

`manter_mais_recente` aplica deduplicação defensiva por chave, mantendo o registro com maior `timestamp_ingestion`. A regra protege a Silver contra reprocessamentos ou cargas duplicadas.

`registrar_dq` grava métricas de execução em `bronze.dq_log`, incluindo tabela, etapa, volumetria, duração e `execucao_id`.

In [0]:
def manter_mais_recente(df: DataFrame, chave: str) -> DataFrame:
    janela = Window.partitionBy(chave).orderBy(F.col("timestamp_ingestion").desc())
    return (
        df.withColumn("_rn", F.row_number().over(janela))
          .filter(F.col("_rn") == 1)
          .drop("_rn")
    )


def registrar_dq(tabela: str, etapa: str, registros: int, duracao: float):
    # Auditoria
    log = spark.createDataFrame(
        [(execucao_id, "silver", tabela, etapa, registros, duracao, None, datetime.now())],
        schema=StructType([
            StructField("execucao_id",      StringType(),    False),
            StructField("camada",           StringType(),    False),
            StructField("tabela",           StringType(),    False),
            StructField("etapa",            StringType(),    False),
            StructField("registros",        LongType(),      True),
            StructField("duracao_segundos", DoubleType(),    True),
            StructField("arquivo_origem",   StringType(),    True),
            StructField("timestamp_log",    TimestampType(), False),
        ])
    )
    log.write.format("delta").mode("append").saveAsTable(f"{catalogo}.bronze.dq_log")

print("Funções utilitárias definidas: manter_mais_recente, registrar_dq")

## 3. Domínios oficiais e variações conhecidas

Define os domínios usados na padronização de categoria, status e método de pagamento.

Os valores canônicos representam a forma normalizada do domínio. Exemplo: `aprovado`, `cartao`, `eletronicos`.

Os aliases representam abreviações conhecidas da origem. Exemplo: `apr -> aprovado`, `crt -> cartao`, `brinq -> brinquedos`.

Erros de digitação não entram nos aliases. Eles são tratados depois pelo fuzzy matching. Essa separação mantém o dicionário menor e evita transformar ruído em regra fixa.

Também são definidos os mapas auxiliares de booleanos e quantidades por extenso, usados nos campos `ativo` e `quantidade`.

In [0]:
CATEGORIA_CANONICAS = [
    "esportes",
    "automotivo",
    "beleza",
    "brinquedos",
    "casa",
    "eletronicos",
    "moveis",
    "vestuario",
]

STATUS_CANONICOS = [
    "aprovado",
    "recusado",
    "reembolsado",
    "processando",
]

PAGAMENTO_CANONICOS = [
    "pix",
    "cartao",
    "boleto",
]

CAPITALIZACAO_CATEGORIA = {
    "esportes": "Esportes",
    "automotivo": "Automotivo",
    "beleza": "Beleza",
    "brinquedos": "Brinquedos",
    "casa": "Casa",
    "eletronicos": "Eletronicos",
    "moveis": "Moveis",
    "vestuario": "Vestuario",
}

CAPITALIZACAO_STATUS = {
    "aprovado": "Aprovado",
    "recusado": "Recusado",
    "reembolsado": "Reembolsado",
    "processando": "Processando",
}

CAPITALIZACAO_PAGAMENTO = {
    "pix": "PIX",
    "cartao": "Cartao",
    "boleto": "Boleto",
}

# Aliases são valores semanticamente abreviados
# Erros pequenos de digitação ficam para o fuzzy
CATEGORIA_ALIASES = {
    "aut": "automotivo",
    "autom": "automotivo",
    "bel": "beleza",
    "belz": "beleza",
    "elet": "eletronicos",
    "esp": "esportes",
    "esport": "esportes",
    "mov": "moveis",
    "cas": "casa",
    "brin": "brinquedos",
    "brinq": "brinquedos",
    "vest": "vestuario",
    "vestu": "vestuario",
}

STATUS_ALIASES = {
    "apr": "aprovado",
    "aprov": "aprovado",
    "rec": "recusado",
    "recus": "recusado",
    "reemb": "reembolsado",
    "reembolso": "reembolsado",
    "proc": "processando",
    "process": "processando",
}

PAGAMENTO_ALIASES = {
    "bol": "boleto",
    "crt": "cartao",
}

BOOL_MAP = {
    "s": True,
    "sim": True,
    "1": True,
    "true": True,
    "yes": True,
    "y": True,
    "verdadeiro": True,

    "n": False,
    "nao": False,
    "não": False,
    "0": False,
    "false": False,
    "no": False,
    "falso": False,
}

QUANTIDADE_EXTENSO = {
    "um": "1",
    "uma": "1",
    "dois": "2",
    "duas": "2",
    "tres": "3",
    "três": "3",
    "quatro": "4",
    "cinco": "5",
    "seis": "6",
    "sete": "7",
    "oito": "8",
    "nove": "9",
    "dez": "10",
}

STATUS_VALIDOS = [CAPITALIZACAO_STATUS[c] for c in STATUS_CANONICOS]
PAGAMENTOS_VALIDOS = [CAPITALIZACAO_PAGAMENTO[c] for c in PAGAMENTO_CANONICOS]
CATEGORIAS_VALIDAS = [CAPITALIZACAO_CATEGORIA[c] for c in CATEGORIA_CANONICAS]

print("[OK] Domínios canônicos e variações conhecidas carregados")

## 4. Normalização textual e mapeamento por regras

Normaliza textos antes do mapeamento de domínio.

A normalização aplica `trim`, `lower`, remoção de acentos, tratamento de alguns caracteres de leetspeak e remoção de caracteres especiais.

In [0]:
def normalizar_python(valor: str) -> str:
    # Normalização equivalente à usada no Spark.
    # Usada apenas no driver para construir chaves de lookup.
    if valor is None:
        return None

    valor = str(valor).strip().lower()
    valor = unicodedata.normalize("NFKD", valor).encode("ascii", "ignore").decode("ascii")

    leet = str.maketrans({
        "0": "o",
        "1": "i",
        "3": "e",
        "4": "a",
        "@": "a",
        "$": "s",
        "5": "s",
        "7": "t",
        "8": "b",
    })

    valor = valor.translate(leet)
    valor = re.sub(r"[^a-z]", "", valor)

    return valor if valor else None


def normalizar_spark(col):
    # Normalização distribuída Spark
    
    c = F.lower(F.trim(col.cast("string")))

    substituicoes = [
        (r"[áàãâä]", "a"),
        (r"[éèêë]", "e"),
        (r"[íìîï]", "i"),
        (r"[óòõôö]", "o"),
        (r"[úùûü]", "u"),
        (r"[ç]", "c"),

        ("0", "o"),
        ("1", "i"),
        ("3", "e"),
        ("4", "a"),
        (r"@", "a"),
        (r"\$", "s"),
        ("5", "s"),
        ("7", "t"),
        ("8", "b"),
    ]

    for origem, destino in substituicoes:
        c = F.regexp_replace(c, origem, destino)
    c = F.regexp_replace(c, r"[^a-z]", "")

    return F.when(F.length(c) > 0, c).otherwise(F.lit(None))


def criar_create_map(dicionario: dict):
    # Cria expressão Spark create_map a partir de um dict
    pares = []

    for chave, valor in dicionario.items():
        pares.extend([F.lit(chave), F.lit(valor)])

    return F.create_map(*pares)


def criar_lookup_deterministico(canonicos: list, aliases: dict, capitalizacao: dict):
    # Lookup mínimo:
    # canônicos normalizados
    # variações conhecidas explícitas de negócio
    lookup = {}

    for canonico in canonicos:
        chave = normalizar_python(canonico)
        lookup[chave] = capitalizacao[canonico]

    for alias, canonico in aliases.items():
        chave = normalizar_python(alias)
        lookup[chave] = capitalizacao[canonico]

    return lookup


LOOKUP_CATEGORIA = criar_lookup_deterministico(
    CATEGORIA_CANONICAS,
    CATEGORIA_ALIASES,
    CAPITALIZACAO_CATEGORIA,
)

LOOKUP_STATUS = criar_lookup_deterministico(
    STATUS_CANONICOS,
    STATUS_ALIASES,
    CAPITALIZACAO_STATUS,
)

LOOKUP_PAGAMENTO = criar_lookup_deterministico(
    PAGAMENTO_CANONICOS,
    PAGAMENTO_ALIASES,
    CAPITALIZACAO_PAGAMENTO,
)

MAP_CATEGORIA = criar_create_map(LOOKUP_CATEGORIA)
MAP_STATUS = criar_create_map(LOOKUP_STATUS)
MAP_PAGAMENTO = criar_create_map(LOOKUP_PAGAMENTO)

print("[OK] Normalização e mapas determinísticos carregados")
print(f"Categoria: {len(LOOKUP_CATEGORIA)} chaves determinísticas")
print(f"Status: {len(LOOKUP_STATUS)} chaves determinísticas")
print(f"Pagamento: {len(LOOKUP_PAGAMENTO)} chaves determinísticas")

## 5. Resolvedores de Booleanos e Quantidade

Campos com regra própria são tratados fora do pipeline de fuzzy.

`resolver_booleano_spark` converte o campo `ativo` para booleano. Valores como `S`, `Sim` e `1` viram `true`; valores como `N`, `Nao` e `0` viram `false`.

Não há fuzzy para booleano, porque o domínio tem apenas dois valores e uma aproximação textual poderia gerar falso positivo.

`resolver_quantidade_spark` converte quantidade para inteiro. A função aceita inteiros, decimais equivalentes a inteiro e alguns números por extenso em português.

In [0]:
def normalizar_booleano_spark(col):
    # Normalização específica para booleanos.
    c = F.lower(F.trim(col.cast("string")))

    substituicoes = [
        (r"[áàãâä]", "a"),
        (r"[éèêë]", "e"),
        (r"[íìîï]", "i"),
        (r"[óòõôö]", "o"),
        (r"[úùûü]", "u"),
        (r"[ç]", "c"),
    ]

    for origem, destino in substituicoes:
        c = F.regexp_replace(c, origem, destino)

    return c


BOOL_MAP_STRING = {k: str(v) for k, v in BOOL_MAP.items()}
SPARK_MAP_BOOL = criar_create_map(BOOL_MAP_STRING)

def resolver_booleano_spark(col):
    raw = SPARK_MAP_BOOL[normalizar_booleano_spark(col)]

    return (
        F.when(raw == "True", F.lit(True))
         .when(raw == "False", F.lit(False))
         .otherwise(F.lit(None).cast("boolean"))
    )


QTD_MAP = criar_create_map(QUANTIDADE_EXTENSO)

def resolver_quantidade_spark(col):
    # Resolve quantidade em Spark nativo:
    # traduz números por extenso
    # aceita inteiros
    # aceita decimais equivalentes a inteiro, como 2.0
    # rejeita textos não numéricos
    # rejeita decimais não inteiros, como 2.5

    qtd_raw = F.lower(F.trim(col.cast("string")))

    qtd_traduzida = F.coalesce(
        QTD_MAP[qtd_raw],
        qtd_raw
    )

    qtd_traduzida = F.regexp_replace(qtd_traduzida, ",", ".")

    qtd_decimal = (
        F.when(
            qtd_traduzida.rlike(r"^-?\d+(\.\d+)?$"),
            qtd_traduzida.cast("decimal(10,2)")
        )
        .otherwise(F.lit(None).cast("decimal(10,2)"))
    )

    return (
        F.when(
            qtd_decimal == F.floor(qtd_decimal),
            qtd_decimal.cast("int")
        )
        .otherwise(F.lit(None).cast("int"))
    )

print("[OK] Resolvedores de booleano e quantidade carregados.")

## 6. Padronização de categorias, status e pagamento

Esta seção resolve categoria, status e método de pagamento usando duas etapas.

Primeiro, o notebook tenta resolver o valor por regra determinística. Essa etapa cobre valores canônicos e variações conhecidas.

Depois, apenas os valores que não foram resolvidos passam pelo fuzzy matching com `F.levenshtein`.

A ordem é importante. O determinístico é mais previsível e mais barato. O fuzzy fica apenas para erros pequenos que não vale a pena mapear manualmente.

O fuzzy roda sobre valores distintos, não sobre todas as linhas. Assim, se um erro como `Aprovadoo` aparece milhares de vezes, a distância é calculada uma única vez e depois aplicada de volta aos registros.

Thresholds usados:

1. Categoria: distância máxima 2 e similaridade mínima 0.85.
2. Status: distância máxima 2 e similaridade mínima 0.85.
3. Pagamento: distância máxima 1 e similaridade mínima 0.85.

Pagamento usa regra mais conservadora porque possui palavras curtas, como `PIX`, com maior risco de falso positivo.

In [0]:
def aplicar_match_deterministico(
    df: DataFrame,
    coluna_origem: str,
    coluna_destino: str,
    spark_map,
) -> DataFrame:
    # Camada 1:
    # Aplica normalização + create_map Spark.
    norm_col = normalizar_spark(F.col(coluna_origem))
    return (
        df
        .withColumn(f"_{coluna_destino}_norm", norm_col)
        .withColumn(coluna_destino, spark_map[F.col(f"_{coluna_destino}_norm")])
        .withColumn(
            f"{coluna_destino}_origem_tratamento",
            F.when(F.col(coluna_origem).isNull(), F.lit("origem_nula"))
             .when(F.col(coluna_destino).isNotNull(), F.lit("deterministico"))
             .otherwise(F.lit("pendente_fuzzy"))
        )
    )

def aplicar_fuzzy_spark_residual(
    df: DataFrame,
    coluna_origem: str,
    coluna_destino: str,
    dominio_capitalizado: list,
    distancia_maxima: int,
    similaridade_minima: float,
) -> DataFrame:
    # Camada 2:
    # Aplica fuzzy matching Spark nativo apenas nos valores residuais não resolvidos pela camada determinística

    dominio_df = (
        spark
        .createDataFrame([(valor,) for valor in dominio_capitalizado], ["valor_canonico"])
        .withColumn(
            "valor_canonico_norm",
            normalizar_spark(F.col("valor_canonico"))
        )
    )

    # Pega apenas valores distintos que ainda não foram resolvidos
    residuos = (
        df
        .filter(F.col(coluna_destino).isNull())
        .withColumn("_valor_origem_norm", normalizar_spark(F.col(coluna_origem)))
        .filter(F.col("_valor_origem_norm").isNotNull())
        .select("_valor_origem_norm")
        .distinct()
    )

    # Calcula candidatos fuzzy apenas uma vez por valor distinto
    candidatos = (
        residuos
        .crossJoin(F.broadcast(dominio_df))
        .withColumn(
            "_distancia",
            F.levenshtein(
                F.col("_valor_origem_norm"),
                F.col("valor_canonico_norm")
            )
        )
        .withColumn(
            "_maior_tamanho",
            F.greatest(
                F.length(F.col("_valor_origem_norm")),
                F.length(F.col("valor_canonico_norm"))
            )
        )
        .withColumn(
            "_similaridade",
            1 - (F.col("_distancia") / F.col("_maior_tamanho"))
        )
        .filter(
            (F.col("_distancia") <= distancia_maxima) &
            (F.col("_similaridade") >= similaridade_minima)
        )
    )

    # Escolhe o melhor candidato por valor normalizado residual
    janela_melhor = Window.partitionBy("_valor_origem_norm").orderBy(
        F.col("_distancia").asc(),
        F.col("_similaridade").desc(),
        F.col("valor_canonico").asc()
    )

    melhores = (
        candidatos
        .withColumn("_rank", F.row_number().over(janela_melhor))
        .filter(F.col("_rank") == 1)
        .select(
            "_valor_origem_norm",
            F.col("valor_canonico").alias(f"{coluna_destino}_fuzzy"),
            F.col("_distancia").alias(f"{coluna_destino}_fuzzy_distancia"),
            F.round(F.col("_similaridade"), 4).alias(f"{coluna_destino}_fuzzy_similaridade"),
        )
    )

    # Junta o match fuzzy de volta na base completa
    resultado = (
        df
        .withColumn("_valor_origem_norm", normalizar_spark(F.col(coluna_origem)))
        .join(melhores, on="_valor_origem_norm", how="left")
        .withColumn(
            coluna_destino,
            F.coalesce(
                F.col(coluna_destino),
                F.col(f"{coluna_destino}_fuzzy")
            )
        )
        .withColumn(
            f"{coluna_destino}_origem_tratamento",
            F.when(
                F.col(f"{coluna_destino}_fuzzy").isNotNull(),
                F.lit("fuzzy")
            )
            .when(
                F.col(coluna_destino).isNotNull(),
                F.col(f"{coluna_destino}_origem_tratamento")
            )
            .when(
                F.col(coluna_origem).isNull(),
                F.lit("origem_nula")
            )
            .otherwise(F.lit("invalido"))
        )
        .drop("_valor_origem_norm")
    )

    return resultado

def resolver_enum_producao(
    df: DataFrame,
    coluna_origem: str,
    coluna_destino: str,
    spark_map,
    dominio_capitalizado: list,
    distancia_maxima: int = 2,
    similaridade_minima: float = 0.85,
) -> DataFrame:
    # 1. determinístico mínimo
    # 2. fuzzy residual Spark
    # 3. auditoria por colunas auxiliares
    df = aplicar_match_deterministico(
        df=df,
        coluna_origem=coluna_origem,
        coluna_destino=coluna_destino,
        spark_map=spark_map,
    )

    df = aplicar_fuzzy_spark_residual(
        df=df,
        coluna_origem=coluna_origem,
        coluna_destino=coluna_destino,
        dominio_capitalizado=dominio_capitalizado,
        distancia_maxima=distancia_maxima,
        similaridade_minima=similaridade_minima,
    )

    return df


print("[OK] Resolvedor de produção carregado: determinístico + fuzzy Spark residual")

## 7. Auditoria de mapeamentos

Persiste o "antes/depois" de cada valor padronizado em `silver.enum_mapeamento_auditoria`. Saída agregada por valor original e por origem de tratamento, sendo uma linha por combinação distinta, não uma por registro de origem.

**Idempotência:** antes de cada append, executa `DELETE` seletivo por `execucao_id + tabela + dominio`. Re-rodar o notebook na mesma execução não duplica auditoria.

**Schema persistido:** valor original, valor mapeado, origem do tratamento (`deterministico`, `fuzzy`, `origem_nula`, `invalido`), distância e similaridade fuzzy quando aplicável, e frequência. Permite responder na apresentação: "quantos registros viraram `Aprovado` por fuzzy, vindos de qual variante, com que distância".

In [0]:
def auditar_mapeamento_enum(
    df: DataFrame,
    tabela: str,
    coluna_original: str,
    coluna_mapeada: str,
    dominio: str
):
    """
    Auditoria de mapeamentos de enumerações.
    Registra:
    - valor original
    - valor mapeado
    - origem do tratamento
    - distância fuzzy
    - similaridade fuzzy
    - frequência
    """

    col_origem = f"{coluna_mapeada}_origem_tratamento"
    col_dist = f"{coluna_mapeada}_fuzzy_distancia"
    col_sim = f"{coluna_mapeada}_fuzzy_similaridade"

    df_auditoria = df

    if col_origem not in df_auditoria.columns:
        df_auditoria = df_auditoria.withColumn(col_origem, F.lit(None).cast("string"))

    if col_dist not in df_auditoria.columns:
        df_auditoria = df_auditoria.withColumn(col_dist, F.lit(None).cast("int"))

    if col_sim not in df_auditoria.columns:
        df_auditoria = df_auditoria.withColumn(col_sim, F.lit(None).cast("double"))

    auditoria = (
        df_auditoria
        .groupBy(
            F.col(coluna_original).cast("string").alias("valor_original"),
            F.col(coluna_mapeada).cast("string").alias("valor_mapeado"),
            F.col(col_origem).alias("origem_tratamento"),
            F.col(col_dist).alias("fuzzy_distancia"),
            F.col(col_sim).alias("fuzzy_similaridade"),
        )
        .agg(F.count("*").alias("qtd"))
        .withColumn("execucao_id", F.lit(execucao_id))
        .withColumn("camada", F.lit("silver"))
        .withColumn("tabela", F.lit(tabela))
        .withColumn("dominio", F.lit(dominio))
        .withColumn("coluna_original", F.lit(coluna_original))
        .withColumn("coluna_mapeada", F.lit(coluna_mapeada))
        .withColumn(
            "status_resolucao",
            F.when(F.col("valor_original").isNull(), F.lit("origem_nula"))
             .when(F.col("valor_mapeado").isNull(), F.lit("nao_mapeado"))
             .otherwise(F.lit("mapeado"))
        )
        .withColumn("timestamp_log", F.current_timestamp())
        .select(
            "execucao_id",
            "camada",
            "tabela",
            "dominio",
            "coluna_original",
            "coluna_mapeada",
            "valor_original",
            "valor_mapeado",
            "origem_tratamento",
            "status_resolucao",
            "fuzzy_distancia",
            "fuzzy_similaridade",
            "qtd",
            "timestamp_log",
        )
    )

    audit_table = f"{catalogo}.silver.enum_mapeamento_auditoria"

    if spark.catalog.tableExists(audit_table):
        spark.sql(f"""
            DELETE FROM {audit_table}
            WHERE execucao_id = '{execucao_id}'
              AND tabela = '{tabela}'
              AND dominio = '{dominio}'
        """)

    (
        auditoria.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(audit_table)
    )

    nao_mapeados = auditoria.filter(F.col("status_resolucao") == "nao_mapeado").count()
    qtd_fuzzy = auditoria.filter(F.col("origem_tratamento") == "fuzzy").agg(F.sum("qtd")).collect()[0][0]

    qtd_fuzzy = qtd_fuzzy or 0

    if nao_mapeados > 0:
        print(f"[WARN] {tabela}.{dominio}: existem valores não mapeados. Ver {audit_table}.")
    else:
        print(f"[OK] {tabela}.{dominio}: todos os valores não nulos foram mapeados.")

    print(f"[INFO] {tabela}.{dominio}: registros resolvidos por fuzzy = {qtd_fuzzy:,}")

## 8. Silver `silver.dim_produtos`


Cria a dimensão Silver de produtos.

A decisão da dimensão é preservar todos os produtos. Registros com problema cadastral continuam na tabela principal, mas recebem flags de qualidade.

Essa decisão evita perda de histórico, porque um produto inativo, sem categoria ou com preço inválido ainda pode estar associado a pedidos antigos.

Tratamentos aplicados:

1. padronização de categoria;
2. conversão de `ativo` para booleano;
3. limpeza e conversão de preço;
4. `trim` em nome do produto e fornecedor;
5. conversão de peso, estoque e data de cadastro;
6. remoção de `avaliacao_interna`.

Flags criadas: categoria_invalida, ativo_invalido e preco_invalido

In [0]:
inicio = time.time()

bronze_produtos = spark.table(f"{catalogo}.bronze.tb_produtos")
qtd_origem = bronze_produtos.count()
registrar_dq("dim_produtos", "leitura_bronze", qtd_origem, time.time() - inicio)
print(f"bronze.tb_produtos: {qtd_origem:,} registros")

# Deduplicação defensiva: remove duplicatas de id_produto mantendo a mais recente
dup_produtos = (
    bronze_produtos.groupBy("id_produto").count()
    .filter(F.col("count") > 1).count()
)
registrar_dq("dim_produtos", "duplicados_bronze", dup_produtos, time.time() - inicio)

if dup_produtos > 0:
    bronze_produtos = manter_mais_recente(bronze_produtos, "id_produto")
    print(f"[WARN] {dup_produtos} id_produto duplicados removidos por timestamp_ingestion")
else:
    print("[OK] Sem duplicatas de id_produto na Bronze")


In [0]:
inicio = time.time()

df_produtos = (
    bronze_produtos
    .drop("avaliacao_interna")
    .withColumn(
        "categoria_original",
        F.when(
            F.lower(F.trim(F.col("categoria").cast("string"))).isin(
                "", "null", "nan", "none", "n/a", "na", "-"
            ),
            None
        ).otherwise(F.col("categoria"))
    )
    .withColumn("ativo_original", F.col("ativo"))
    .withColumn("preco_original", F.col("preco"))
)

# Categoria: determinístico mínimo + fuzzy Spark residual
df_produtos = resolver_enum_producao(
    df=df_produtos,
    coluna_origem="categoria_original",
    coluna_destino="categoria",
    spark_map=MAP_CATEGORIA,
    dominio_capitalizado=CATEGORIAS_VALIDAS,
    distancia_maxima=2,
    similaridade_minima=0.85,
)

# Ativo: regra determinística, sem fuzzy
df_produtos = df_produtos.withColumn(
    "ativo",
    resolver_booleano_spark(F.col("ativo_original"))
)

# Preço: limpeza numérica
df_produtos = (
    df_produtos
    .withColumn(
        "preco_limpo",
        F.when(
            F.lower(F.trim(F.col("preco").cast("string"))).isin(
                "", "null", "nan", "none", "n/a", "na", "-"
            ),
            None
        ).otherwise(F.col("preco").cast("string"))
    )
    .withColumn("preco_limpo", F.regexp_replace(F.col("preco_limpo"), r"R\$\s*", ""))
    .withColumn("preco_limpo", F.regexp_replace(F.col("preco_limpo"), r",", "."))
    .withColumn("preco", F.expr("try_cast(preco_limpo as decimal(10,2))"))
    .drop("preco_limpo")
)

auditar_mapeamento_enum(
    df=df_produtos,
    tabela="dim_produtos",
    coluna_original="categoria_original",
    coluna_mapeada="categoria",
    dominio="categoria",
)

qtd_pos_normalizacao = df_produtos.count()

registrar_dq(
    "dim_produtos",
    "normalizacao",
    qtd_pos_normalizacao,
    time.time() - inicio
)

print(f"Após normalização: {qtd_pos_normalizacao:,} registros")

In [0]:
# Predicados de validade dim_produtos

PRED_CATEGORIA_INVALIDA = F.col("categoria").isNull()
PRED_ATIVO_INVALIDO     = F.col("ativo").isNull()
PRED_PRECO_INVALIDO     = F.col("preco").isNull() | (F.col("preco") <= 0)

PRED_PRODUTO_INVALIDO = (
    PRED_CATEGORIA_INVALIDA |
    PRED_ATIVO_INVALIDO     |
    PRED_PRECO_INVALIDO
)

print("[OK] Predicados dim_produtos definidos")

In [0]:
inicio = time.time()

df_invalidos_produtos = df_produtos.filter(
    PRED_PRODUTO_INVALIDO
).withColumn(
    "motivo_invalido",
    F.when(PRED_CATEGORIA_INVALIDA, F.lit("categoria_nao_resolvida"))
     .when(PRED_ATIVO_INVALIDO,     F.lit("ativo_nao_reconhecido"))
     .when(F.col("preco").isNull(), F.lit("preco_nao_numerico"))
     .when(F.col("preco") <= 0,     F.lit("preco_nao_positivo"))
     .otherwise(F.lit("motivo_nao_classificado"))
)

qtd_invalidos = df_invalidos_produtos.count()

if qtd_invalidos > 0:
    print("Distribuição de motivos para produtos com inconsistência cadastral:")
    (
        df_invalidos_produtos
        .groupBy("motivo_invalido")
        .count()
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

registrar_dq("dim_produtos", "registros_com_flag", qtd_invalidos, time.time() - inicio)

print(
    f"Produtos com alguma flag de qualidade: {qtd_invalidos:,} "
    f"(serão consultáveis via silver.dim_produtos_invalidos)")

In [0]:
inicio = time.time()

df_produtos_silver = (
    df_produtos
    .withColumn("nome_produto", F.trim(F.col("nome_produto")))
    .withColumn("fornecedor", F.trim(F.col("fornecedor")))
    .withColumn("categoria_invalida", PRED_CATEGORIA_INVALIDA)
    .withColumn("ativo_invalido", PRED_ATIVO_INVALIDO)
    .withColumn("preco_invalido", PRED_PRECO_INVALIDO)
    .withColumn("peso_kg", F.expr("try_cast(peso_kg as decimal(10,2))"))
    .withColumn("estoque_disponivel", F.expr("try_cast(estoque_disponivel as int)"))
    .withColumn("data_cadastro_produto", F.to_date(F.col("data_cadastro_produto"), "yyyy-MM-dd"))
    .select(
        "id_produto",
        "nome_produto",
        "categoria",
        "preco",
        "ativo",
        "fornecedor",
        "peso_kg",
        "estoque_disponivel",
        "data_cadastro_produto",
        "categoria_original",
        "categoria_origem_tratamento",
        "categoria_fuzzy_distancia",
        "categoria_fuzzy_similaridade",
        "categoria_invalida",
        "ativo_invalido",
        "preco_invalido",
        "timestamp_ingestion",
        "arquivo_origem",
    )
)

(
    df_produtos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.silver.dim_produtos")
)

# Exposição dos produtos com inconsistência cadastral como VIEW
# A dimensão principal preserva todos os produtos; a view apenas facilita auditoria
spark.sql(f"DROP TABLE IF EXISTS {catalogo}.silver.dim_produtos_invalidos")

spark.sql(f"""
    CREATE OR REPLACE VIEW {catalogo}.silver.dim_produtos_invalidos AS
    SELECT
        *,
        CASE
            WHEN categoria_invalida THEN 'categoria_nao_resolvida'
            WHEN ativo_invalido     THEN 'ativo_nao_reconhecido'
            WHEN preco IS NULL      THEN 'preco_nao_numerico'
            WHEN preco <= 0         THEN 'preco_nao_positivo'
            ELSE 'motivo_nao_classificado'
        END AS motivo_invalido
    FROM {catalogo}.silver.dim_produtos
    WHERE categoria_invalida
       OR ativo_invalido
       OR preco_invalido
""")

print("View silver.dim_produtos_invalidos criada sobre silver.dim_produtos.")

qtd_silver_produtos = df_produtos_silver.count()
registrar_dq("dim_produtos", "escrita_silver", qtd_silver_produtos, time.time() - inicio)

print(f"silver.dim_produtos: {qtd_silver_produtos:,} produtos mantidos")

## 9. Silver `silver.fat_pedidos`

Cria a fato Silver de pedidos.

Pedidos válidos entram em `silver.fat_pedidos`. Pedidos com problema crítico entram em `silver.fat_pedidos_invalidos`.

A regra é mais rígida do que em produtos porque pedido é fato transacional. Ele afeta receita, ticket médio, quantidade vendida, status de venda e performance de produto.

Tratamentos aplicados:

1. padronização de status;
2. padronização de método de pagamento;
3. conversão de quantidade;
4. limpeza de `valor_pedido`;
5. cálculo de `valor_unitario`;
6. parser de datas em múltiplos formatos;
7. aplicação do intervalo operacional.

`valor_pedido` é usado como valor total histórico da transação. A receita não é recalculada a partir do preço atual do catálogo.

O predicado `PRED_PEDIDO_VALIDO` concentra as regras de validade e é usado tanto para gerar a fato final quanto para separar a quarentena.

Após a normalização, o DataFrame de pedidos é materializado em uma stage Delta. Essa etapa evita reprocessar a normalização nas células seguintes.

In [0]:
inicio = time.time()

bronze_pedidos = spark.table(f"{catalogo}.bronze.tb_pedidos")
qtd_origem_pedidos = bronze_pedidos.count()
registrar_dq("fat_pedidos", "leitura_bronze", qtd_origem_pedidos, time.time() - inicio)
print(f"bronze.tb_pedidos: {qtd_origem_pedidos:,} registros")

# Deduplicação defensiva: remove duplicatas de id_pedido mantendo a mais recente
dup_pedidos = (
    bronze_pedidos.groupBy("id_pedido").count()
    .filter(F.col("count") > 1).count()
)
registrar_dq("fat_pedidos", "duplicados_bronze", dup_pedidos, time.time() - inicio)

if dup_pedidos > 0:
    bronze_pedidos = manter_mais_recente(bronze_pedidos, "id_pedido")
    print(f"[WARN] {dup_pedidos} id_pedido duplicados removidos por timestamp_ingestion")
else:
    print("[OK] Sem duplicatas de id_pedido na Bronze")


In [0]:
inicio = time.time()

# data_pedido possui múltiplos formatos no source.
# A cascata abaixo tenta parsear sem quebrar; formatos não reconhecidos viram null e vão para inválidos.
df_pedidos = (
    bronze_pedidos
    .withColumn("status_original",            F.col("status"))
    .withColumn("metodo_pagamento_original",   F.col("metodo_pagamento"))
    .withColumn("valor_pedido_original",       F.col("valor_pedido"))
    .withColumn("quantidade_original",         F.col("quantidade"))
    .withColumn("data_pedido_original",        F.col("data_pedido"))

    # Valor
    .withColumn(
        "valor_limpo",
        F.when(F.lower(F.trim(F.col("valor_pedido").cast("string"))) == "null", None)
        .otherwise(F.col("valor_pedido").cast("string"))
    )
    .withColumn("valor_limpo", F.regexp_replace(F.col("valor_limpo"), r"R\$\s*", ""))
    .withColumn("valor_limpo", F.regexp_replace(F.col("valor_limpo"), r",", "."))
    .withColumn("valor_pedido", F.expr("try_cast(valor_limpo as decimal(10,2))"))
    .drop("valor_limpo")

    # Quantidade:
    .withColumn("quantidade", resolver_quantidade_spark(F.col("quantidade")))

    # Data: parser em cascata multi-formato
    .withColumn(
        "data_swapped",
        F.when(
            F.col("data_pedido").rlike(r"^\d{4}/(1[3-9]|2[0-9]|3[01])/\d{1,2}$"),
            F.regexp_replace(
                F.col("data_pedido"),
                r"^(\d{4})/(\d{2})/(\d{1,2})$",
                "$1-$3-$2"
            )
        ).otherwise(F.col("data_pedido"))
    )
    .withColumn(
        "data_pedido",
        F.coalesce(
            F.to_date(F.expr("try_to_timestamp(data_swapped, 'yyyy-MM-dd')")),
            F.to_date(F.expr("try_to_timestamp(data_swapped, 'dd/MM/yyyy')")),
            F.to_date(F.expr("try_to_timestamp(data_swapped, 'dd-MM-yyyy')")),
            F.to_date(F.expr("try_to_timestamp(data_swapped, 'yyyy/MM/dd')")),
            F.to_date(F.expr("try_to_timestamp(data_swapped, 'MM-dd-yyyy')"))
        )
    )
    .drop("data_swapped")
)

# Status: determinístico mínimo + fuzzy Spark residual
df_pedidos = resolver_enum_producao(
    df=df_pedidos,
    coluna_origem="status_original",
    coluna_destino="status",
    spark_map=MAP_STATUS,
    dominio_capitalizado=STATUS_VALIDOS,
    distancia_maxima=2,
    similaridade_minima=0.85,
)

# Pagamento: determinístico mínimo + fuzzy mais conservador
df_pedidos = resolver_enum_producao(
    df=df_pedidos,
    coluna_origem="metodo_pagamento_original",
    coluna_destino="metodo_pagamento",
    spark_map=MAP_PAGAMENTO,
    dominio_capitalizado=PAGAMENTOS_VALIDOS,
    distancia_maxima=1,
    similaridade_minima=0.85,
)

stage_pedidos = f"{catalogo}.silver._stg_fat_pedidos_normalizado"

(
    df_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(stage_pedidos)
)

df_pedidos = spark.table(stage_pedidos)

qtd_pos_normalizacao_pedidos = df_pedidos.count()

registrar_dq(
    "fat_pedidos",
    "normalizacao",
    qtd_pos_normalizacao_pedidos,
    time.time() - inicio
)

print(
    f"Após normalização: {qtd_pos_normalizacao_pedidos:,} registros "
    f"(materializados em {stage_pedidos})"
)

In [0]:
# Predicado de validade: fat_pedidos
# filter(~PRED_PEDIDO_VALIDO) → fat_pedidos_invalidos
# filter(PRED_PEDIDO_VALIDO) → df_pedidos_validos

PRED_PEDIDO_VALIDO = (
    F.col("id_pedido").isNotNull() &
    F.col("id_cliente").isNotNull() &
    F.col("id_produto").isNotNull() &
    F.col("status").isNotNull() &
    F.col("metodo_pagamento").isNotNull() &
    F.col("valor_pedido").isNotNull() &
    (F.col("valor_pedido") > 0) &
    F.col("quantidade").isNotNull() &
    (F.col("quantidade") > 0) &
    F.col("data_pedido").isNotNull() &
    (F.col("data_pedido") >= F.to_date(F.lit(DATA_MIN_OPERACAO))) &
    (F.col("data_pedido") <= DATA_REFERENCIA_COL)
)

print("[OK] Predicado fat_pedidos definido")

In [0]:
# Auditoria dos resolvedores fuzzy/enum sobre todos os pedidos normalizados
inicio = time.time()

auditar_mapeamento_enum(
    df=df_pedidos,
    tabela="fat_pedidos",
    coluna_original="status_original",
    coluna_mapeada="status",
    dominio="status"
)

auditar_mapeamento_enum(
    df=df_pedidos,
    tabela="fat_pedidos",
    coluna_original="metodo_pagamento_original",
    coluna_mapeada="metodo_pagamento",
    dominio="metodo_pagamento"
)

registrar_dq(
    tabela="fat_pedidos",
    etapa="auditoria_mapeamentos",
    registros=df_pedidos.count(),
    duracao=time.time() - inicio
)

print("[OK] Auditoria de mapeamentos de todos os pedidos normalizados registrada")

In [0]:
inicio = time.time()

df_invalidos_pedidos = df_pedidos.filter(
    ~PRED_PEDIDO_VALIDO
).withColumn(
    "motivo_invalido",
    F.when(F.col("id_pedido").isNull(),         F.lit("id_pedido_nulo"))
     .when(F.col("id_cliente").isNull(),        F.lit("id_cliente_nulo"))
     .when(F.col("id_produto").isNull(),        F.lit("id_produto_nulo"))
     .when(F.col("status").isNull(),            F.lit("status_sem_match_canonico"))
     .when(F.col("metodo_pagamento").isNull(),  F.lit("pagamento_sem_match_canonico"))
     .when(F.col("valor_pedido").isNull(),      F.lit("valor_nao_numerico"))
     .when(F.col("valor_pedido") <= 0,          F.lit("valor_nao_positivo"))
     .when(F.col("quantidade").isNull(),        F.lit("quantidade_nao_numerica"))
     .when(F.col("quantidade") <= 0,            F.lit("quantidade_nao_positiva"))
     .when(F.col("data_pedido").isNull(),       F.lit("data_invalida"))
     .when(
         F.col("data_pedido") < F.to_date(F.lit(DATA_MIN_OPERACAO)),
         F.lit("data_anterior_operacao")
     )
     .when(
         F.col("data_pedido") > DATA_REFERENCIA_COL,
         F.lit("data_futura")
     )
     .otherwise(F.lit("motivo_nao_classificado"))
)

# Seleção explícita para manter a quarentena limpa e legível
# Guardamos o valor original da Bronze, o valor tratado na Silver e o motivo principal
df_invalidos_pedidos = df_invalidos_pedidos.select(
    "id_pedido",
    "id_cliente",
    "id_produto",
    # Valores originais da Bronze
    "data_pedido_original",
    "valor_pedido_original",
    "quantidade_original",
    "status_original",
    "metodo_pagamento_original",
    # Valores tratados na Silver
    "data_pedido",
    "valor_pedido",
    "quantidade",
    "status",
    "metodo_pagamento",
    # Auditoria
    "motivo_invalido",
    "timestamp_ingestion",
    "arquivo_origem",
)

qtd_invalidos_pedidos = df_invalidos_pedidos.count()

if qtd_invalidos_pedidos > 0:
    print("Distribuição de motivos: pedidos inválidos:")
    (
        df_invalidos_pedidos
        .groupBy("motivo_invalido")
        .count()
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

(
    df_invalidos_pedidos.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.silver.fat_pedidos_invalidos")
)

registrar_dq(
    "fat_pedidos",
    "registros_invalidos",
    qtd_invalidos_pedidos,
    time.time() - inicio
)

print(f"silver.fat_pedidos_invalidos: {qtd_invalidos_pedidos:,} registros")

In [0]:
df_pedidos_validos = df_pedidos.filter(PRED_PEDIDO_VALIDO)

inicio = time.time()

# Na origem, valor_pedido representa o valor total do pedido/item.
# Para atender ao contrato analítico, derivamos valor_unitario a partir de valor_total / quantidade.
df_pedidos_silver = df_pedidos_validos.select(
    F.col("id_pedido"),
    F.col("id_cliente"),
    F.col("id_produto"),
    F.col("data_pedido"),
    F.col("quantidade"),
    (F.col("valor_pedido") / F.col("quantidade")).cast("decimal(12,2)").alias("valor_unitario"),
    F.col("valor_pedido").cast("decimal(12,2)").alias("valor_total"),
    F.col("status"),
    F.col("metodo_pagamento"),
    F.col("timestamp_ingestion"),
    F.col("arquivo_origem"),
)

qtd_silver_pedidos = df_pedidos_silver.count()
registrar_dq("fat_pedidos", "preparacao_silver", qtd_silver_pedidos, time.time() - inicio)

(
    df_pedidos_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalogo}.silver.fat_pedidos")
)

registrar_dq("fat_pedidos", "escrita_silver", qtd_silver_pedidos, time.time() - inicio)

print(f"silver.fat_pedidos: {qtd_silver_pedidos:,} registros válidos")
print(f"Taxa de descarte: {(qtd_origem_pedidos - qtd_silver_pedidos) / qtd_origem_pedidos * 100:.2f}%")

# Limpeza da stage
spark.sql(f"DROP TABLE IF EXISTS {stage_pedidos}")
print(f"[OK] Stage transitória removida: {stage_pedidos}")

## 10. Otimização Delta

Executa `OPTIMIZE` com `ZORDER` nas tabelas Silver finais.

`silver.fat_pedidos` é otimizada por:
data_pedido, status, id_produto e id_cliente

In [0]:
# Otimização física Delta Lake melhora performance de queries analíticas
# ZORDER co-localiza dados pelas colunas mais usadas em filtros e joins da Gold
inicio = time.time()

tabelas_otimizar = {
    f"{catalogo}.silver.fat_pedidos":  ["data_pedido", "status", "id_produto", "id_cliente"],
    f"{catalogo}.silver.dim_produtos": ["id_produto"],
}

for tabela, colunas in tabelas_otimizar.items():
    try:
        if spark.catalog.tableExists(tabela):
            colunas_zorder = ", ".join(colunas)
            spark.sql(f"OPTIMIZE {tabela} ZORDER BY ({colunas_zorder})")
            print(f"[OK] OPTIMIZE executado: {tabela} ZORDER BY ({colunas_zorder})")
        else:
            print(f"[SKIP] Tabela não encontrada para OPTIMIZE: {tabela}")

    except Exception as e:
        print(f"[WARN] OPTIMIZE não executado para {tabela}: {str(e)}")

registrar_dq(
    tabela="pedidos_produtos",
    etapa="optimize_zorder",
    registros=len(tabelas_otimizar),
    duracao=time.time() - inicio
)

## 11. Validação da Camada Silver

Validações executadas:

1. tabelas existem e possuem registros;
2. schemas esperados estão presentes;
3. chaves primárias não têm nulos;
4. chaves primárias não têm duplicatas;
5. chaves obrigatórias de pedido estão preenchidas;
6. domínios de status, pagamento e categoria estão corretos;
7. tipos físicos batem com o contrato esperado;
8. `valor_unitario * quantidade` é coerente com `valor_total`;
9. pedidos possuem produto correspondente;
10. pedidos reconciliam entre Bronze, válidos e inválidos;
11. produtos reconciliam entre Bronze e Silver;
12. datas respeitam o intervalo operacional;
13. flags de preço inválido estão coerentes;
14. auditoria não possui valores críticos sem mapeamento.

In [0]:
def check_existencia_e_volume(tabela_full: str):
    # Retorna (df, total) para reuso
    if not spark.catalog.tableExists(tabela_full):
        falhas.append(f"{tabela_full} ausente")
        return None, 0
    df    = spark.table(tabela_full)
    total = df.count()
    if total == 0:
        falhas.append(f"{tabela_full} vazia")
    else:
        print(f"[OK] {tabela_full}: {total:,} registros")
    return df, total


def check_schema(df, esperadas: set, nome_tabela: str):
    # Verifica que todas as colunas da Silver estão presentes
    faltantes = esperadas - set(df.columns)
    if faltantes:
        falhas.append(f"{nome_tabela}: colunas ausentes {sorted(faltantes)}")
    else:
        print(f"[OK] {nome_tabela}: schema completo")


def check_nao_nulo(df, coluna: str, nome_tabela: str):
    # Usado em PKs e FKs obrigatórias
    nulos = df.filter(F.col(coluna).isNull()).count()
    if nulos > 0:
        falhas.append(f"{nome_tabela}.{coluna}: {nulos} nulos em coluna obrigatória")
    else:
        print(f"[OK] {nome_tabela}.{coluna}: sem nulos")


def check_sem_duplicatas(df, coluna: str, nome_tabela: str):
    dup = df.groupBy(coluna).count().filter(F.col("count") > 1).count()
    if dup > 0:
        falhas.append(f"{nome_tabela}.{coluna}: {dup} duplicatas")
    else:
        print(f"[OK] {nome_tabela}.{coluna}: sem duplicatas")


def check_enum(df, coluna: str, dominio: list, nome_tabela: str):
    # Verifica conformidade com o domínio canônico
    fora = df.filter(~F.col(coluna).isin(dominio)).count()
    if fora > 0:
        falhas.append(f"{nome_tabela}.{coluna}: {fora} valores fora do domínio")
    else:
        print(f"[OK] {nome_tabela}.{coluna}: domínio respeitado")


def check_tipos(df, esperados: dict, nome_tabela: str):
    # Verifica tipos reais com da Silver
    reais = {f.name: f.dataType.simpleString() for f in df.schema.fields}
    erros = [(c, reais.get(c), t) for c, t in esperados.items() if reais.get(c) != t]
    if erros:
        for col, real, esp in erros:
            falhas.append(f"{nome_tabela}.{col}: tipo real={real}, esperado={esp}")
    else:
        print(f"[OK] {nome_tabela}: tipos conforme contrato")

def check_reconciliacao(qtd_bronze: int, qtd_validos: int, qtd_invalidos: int, nome: str):
    # Garante zero perda: bronze = validos + invalidos
    total = qtd_validos + qtd_invalidos
    if total != qtd_bronze:
        falhas.append(
            f"{nome}: bronze={qtd_bronze:,} ≠ validos={qtd_validos:,} + invalidos={qtd_invalidos:,}"
        )
    else:
        print(f"[OK] {nome}: bronze={qtd_bronze:,} = validos={qtd_validos:,} + invalidos={qtd_invalidos:,}")

def check_reconciliacao_dimensao(qtd_bronze: int, qtd_silver: int, nome: str):
    if qtd_bronze != qtd_silver:
        falhas.append(
            f"{nome}: bronze={qtd_bronze:,} ≠ silver={qtd_silver:,}"
        )
    else:
        print(f"[OK] {nome}: bronze={qtd_bronze:,} = silver={qtd_silver:,}")

def check_range_temporal(df, coluna: str, data_min: str, nome_tabela: str):
    datas_antes_operacao = df.filter(
        F.col(coluna) < F.to_date(F.lit(data_min))
    ).count()

    if datas_antes_operacao > 0:
        falhas.append(
            f"{nome_tabela}.{coluna}: {datas_antes_operacao} datas anteriores a {data_min}"
        )
    else:
        print(f"[OK] {nome_tabela}.{coluna}: datas dentro do limite mínimo")
        
def check_data_maxima(df, coluna: str, data_ref_col, nome_tabela: str):
    futuras = df.filter(F.col(coluna) > data_ref_col).count()

    if futuras > 0:
        falhas.append(
            f"{nome_tabela}.{coluna}: {futuras} datas futuras acima da referência"
        )
    else:
        print(f"[OK] {nome_tabela}.{coluna}: sem datas futuras acima da referência")

def check_coerencia_valor(df, nome_tabela: str, tolerancia: float = 0.05):
    incoerentes = df.filter(
        F.abs(
            (F.col("valor_unitario") * F.col("quantidade")) - F.col("valor_total")
        ) > F.lit(tolerancia)
    ).count()

    if incoerentes > 0:
        falhas.append(
            f"{nome_tabela}: {incoerentes} pedidos com "
            f"|valor_unitario * quantidade - valor_total| > {tolerancia}"
        )
    else:
        print(
            f"[OK] {nome_tabela}: coerência valor_unitario * quantidade ~= valor_total"
        )

def check_invariante_flag(df, col_flag: str, col_valor: str, nome_tabela: str):
    # Flag inválida não pode coexistir com valor positivo
    vazado = df.filter(
        F.col(col_flag) & F.col(col_valor).isNotNull() & (F.col(col_valor) > 0)
    ).count()
    if vazado > 0:
        falhas.append(f"{nome_tabela}: {vazado} com {col_flag}=true mas {col_valor} positivo exposto")
    else:
        print(f"[OK] {nome_tabela}: invariante {col_flag} respeitada")


def check_auditoria_mapeamentos(audit_table: str, filtros_criticos, execucao_id: str):
    # Falha se algum domínio crítico ficou sem match canônico na auditoria fuzzy
    if not spark.catalog.tableExists(audit_table):
        falhas.append(f"{audit_table} ausente")
        return
    nao_mapeados = (
        spark.table(audit_table)
        .filter(F.col("execucao_id") == execucao_id)
        .filter(F.col("status_resolucao") == "nao_mapeado")
        .filter(filtros_criticos)
        .count()
    )
    if nao_mapeados > 0:
        falhas.append(f"Mapeamentos críticos sem resolução: {nao_mapeados}: ver {audit_table}")
    else:
        print("[OK] Auditoria: domínios críticos: todos resolvidos")

print("[OK] Funções de validação carregadas")

In [0]:
print("\nValidação final da camada Silver: produtos e pedidos\n")

falhas = []

# 1. Existência e volumetria
df_prod, qtd_prod = check_existencia_e_volume(f"{catalogo}.silver.dim_produtos")
df_ped,  qtd_ped  = check_existencia_e_volume(f"{catalogo}.silver.fat_pedidos")

if df_prod is None or df_ped is None:
    raise Exception(
        "Validação interrompida: tabelas Silver críticas ausentes.\n- " + "\n- ".join(falhas)
    )

# 2. Schema
check_schema(df_prod, {
    "id_produto",
    "nome_produto",
    "categoria",
    "preco",
    "ativo",
    "fornecedor",
    "peso_kg",
    "estoque_disponivel",
    "data_cadastro_produto",

    "categoria_original",
    "categoria_origem_tratamento",
    "categoria_fuzzy_distancia",
    "categoria_fuzzy_similaridade",

    "categoria_invalida",
    "ativo_invalido",
    "preco_invalido",
    "timestamp_ingestion",
    "arquivo_origem",
}, "dim_produtos")

check_schema(df_ped, {
    "id_pedido", "id_cliente", "id_produto", "data_pedido",
    "quantidade", "valor_unitario", "valor_total",
    "status", "metodo_pagamento",
    "timestamp_ingestion", "arquivo_origem",
}, "fat_pedidos")

# 3. Chaves primárias
check_nao_nulo(df_prod,       "id_produto", "dim_produtos")
check_sem_duplicatas(df_prod, "id_produto", "dim_produtos")

check_nao_nulo(df_ped,       "id_pedido", "fat_pedidos")
check_sem_duplicatas(df_ped, "id_pedido", "fat_pedidos")

# 4. FKs obrigatórias em fat_pedidos
check_nao_nulo(df_ped, "id_cliente", "fat_pedidos")
check_nao_nulo(df_ped, "id_produto", "fat_pedidos")

# 5. Domínios enum
check_enum(df_ped, "status",           STATUS_VALIDOS,     "fat_pedidos")
check_enum(df_ped, "metodo_pagamento", PAGAMENTOS_VALIDOS, "fat_pedidos")
check_enum(
    df_prod.filter(~F.col("categoria_invalida")),
    "categoria", CATEGORIAS_VALIDAS, "dim_produtos"
)

# 6. Tipos contratuais
check_tipos(df_ped, {
    "data_pedido":    "date",
    "quantidade":     "int",
    "valor_unitario": "decimal(12,2)",
    "valor_total":    "decimal(12,2)",
}, "fat_pedidos")

check_tipos(df_prod, {
    "preco":                 "decimal(10,2)",
    "ativo":                 "boolean",
    "peso_kg":               "decimal(10,2)",
    "estoque_disponivel":    "int",
    "data_cadastro_produto": "date",
}, "dim_produtos")

# 6.1. Coerência financeira derivada
check_coerencia_valor(df_ped, "fat_pedidos")

# 7. Integridade referencial
check_fk(df_ped, df_prod, "id_produto", "fat_pedidos", "dim_produtos")

# 8. Reconciliação Bronze ↔ Silver_validos + Silver_invalidos
qtd_inv_ped = spark.table(f"{catalogo}.silver.fat_pedidos_invalidos").count()
check_reconciliacao(
    qtd_bronze    = spark.table(f"{catalogo}.bronze.tb_pedidos").count(),
    qtd_validos   = qtd_ped,
    qtd_invalidos = qtd_inv_ped,
    nome          = "fat_pedidos",
)

# 8.1. Reconciliação de produtos
check_reconciliacao_dimensao(
    qtd_bronze = spark.table(f"{catalogo}.bronze.tb_produtos").count(),
    qtd_silver = qtd_prod,
    nome       = "dim_produtos",
)

# 9. Range temporal de data_pedido
check_range_temporal(df_ped, "data_pedido", "2018-01-01", "fat_pedidos")
check_data_maxima(df_ped, "data_pedido", DATA_REFERENCIA_COL, "fat_pedidos")

# 10. Invariante: preco_invalido=true não pode ter preco positivo exposto
check_invariante_flag(df_prod, "preco_invalido", "preco", "dim_produtos")

# 11. Auditoria de mapeamentos críticos
filtros_criticos = (
    ((F.col("tabela") == "fat_pedidos")  & F.col("dominio").isin("status", "metodo_pagamento")) |
    ((F.col("tabela") == "dim_produtos") & (F.col("dominio") == "categoria"))
)
check_auditoria_mapeamentos(
    audit_table      = f"{catalogo}.silver.enum_mapeamento_auditoria",
    filtros_criticos = filtros_criticos,
    execucao_id      = execucao_id,
)

if falhas:
    raise Exception("Validação Silver falhou:\n- " + "\n- ".join(falhas))

print("\n[SUCESSO] Camada Silver_pedidos_produtos validada")